In [3]:
import gradio as gr
from groq import Groq
import os
from dotenv import load_dotenv

from openai import OpenAI

load_dotenv()

True

In [4]:
groq_api_key = os.getenv('GROQ_API_KEY')
client = Groq(api_key=groq_api_key)

In [5]:
ollama = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama" # We don't actually need a key for local, but the library asks for one
)

In [6]:
if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

Groq API Key exists and begins gsk_


In [10]:
import gradio as gr

# Define the persona
SYSTEM_MESSAGE = "You are a helpful assistant named Umair's AI."

def chat_stream(message, history):
    '''
    message: Current user input string
    history: List of [user_msg, bot_msg] from previous turns
    '''
    
    # 1. Rebuild context from history
    messages_payload = [
        {"role": "system", "content": SYSTEM_MESSAGE}
    ]
    
    # Loop through history to build memory
    for user_msg, bot_msg in history:
        if user_msg:
            messages_payload.append({"role": "user", "content": user_msg})
        if bot_msg:
            messages_payload.append({"role": "assistant", "content": bot_msg})
    
    # Add current message
    messages_payload.append({"role": "user", "content": message})

    # 2. Call API with streaming
    try:
        stream = ollama.chat.completions.create(
            model="llama3.2:1b", 
            messages=messages_payload,
            max_tokens=1024,
            temperature=0.7,
            stream=True
        )

        # 3. Yield chunks for typing effect
        partial_response = ""
        for chunk in stream:
            content = chunk.choices[0].delta.content
            if content:
                partial_response += content
                yield partial_response
                
    except Exception as e:
        yield f"Error: {str(e)}"

In [ ]:
# Create the interface
view = gr.ChatInterface(
    fn=chat_stream,
    title="Groq Memory Chat (Jupyter)",
    description="A streaming, stateful chatbot running inside a notebook.",
    examples=["Hi, I'm Umair", "Explain quantum physics in simple terms"],
    theme="soft"
)

# Launching in a notebook requires preventing the thread from blocking
# .launch(inline=True) shows it inside the notebook cell output
view.launch(inline=True, share=False, debug=True, inbrowser=True)

In [6]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."}
]

print("Chat started! (Type 'exit' to quit)\n")

while True:
    user_input = input("You: ")
    if user_input.lower() == "exit": break

    # Add user message to history
    messages.append({"role": "user", "content": user_input})

    # 2. Call API with stream=True
    stream = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=messages,
        max_tokens=1024,
        temperature=0.7,
        stream=True  # <--- CRITICAL: Turn on streaming
    )

    print("AI: ", end="") # Print the label once
    
    full_response = ""

    # 3. The Streaming Loop
    # instead of waiting for 'response', we loop through 'stream'
    for chunk in stream:
        # Check if there is content (sometimes chunks are just metadata)
        if chunk.choices[0].delta.content is not None:
            
            
            # Get the tiny piece of text
            text_chunk = chunk.choices[0].delta.content
            
            # Print it immediately without a new line
            print(text_chunk, end="", flush=True)
            
            # Save it to our variable so we can add it to history later
            full_response += text_chunk

    print("\n") # Print a new line after the AI is done talking

    # 4. Save the full response to history for the next turn
    messages.append({"role": "assistant", "content": full_response})

Chat started! (Type 'exit' to quit)

AI: Hello! How can I assist you today?

AI: Nice to meet you, Umair! How can I help you today?



In [7]:
import time

In [8]:
# 2. Define the 3 Personalities (System Prompts)

# Agent 1: Llama 3.2 (The Polite One)
prompt_polite = """
You are a very shy, polite, and sweet assistant. 
You always guide with a humble voice. 
You try to be helpful but you are very soft-spoken.
"""

# Agent 2: Qwen 2.5 (The Snarky Bully)
prompt_snarky = """
You are a rude, snarky, and mean AI. 
You think polite people are stupid. 
You degrade the previous speaker for being weak. 
Make fun of them. Be sarcastic.
"""

# Agent 3: Llama 3.2 (The Strict Professional)
prompt_professional = """
You are a strict, no-nonsense manager. 
You hate wasting time. 
You scold the snarky agent for being rude and tell him how to answer.
Force everyone to be professional.
"""

# 3. The Function to get a response
def ask_bot(model_name, system_prompt, user_input):
    response = ollama.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ],
        max_tokens=50
    )
    return response.choices[0].message.content

# --- THE GAME START ---

print("--- STARTING THE CRAZY CHAT ---\n")

# Start with a simple topic
current_message = "Let's discuss how to write Python code."

# We will loop 2 times to see the chaos
for round in range(1, 3):
    print(f"\n=== ROUND {round} ===\n")

    # --- TURN 1: Polite Llama ---
    # It reacts to the current topic
    print(f"🔴 Llama (Polite):")
    response_1 = ask_bot("llama3.2:1b", prompt_polite, current_message)
    print(response_1)
    print("-" * 20)
    time.sleep(1) # Wait a second so we can read

    # --- TURN 2: Snarky Qwen ---
    # It reacts to what Polite Llama just said
    print(f"🔵 Qwen (Snarky):")
    response_2 = ask_bot("qwen2.5:3b", prompt_snarky, response_1)
    print(response_2)
    print("-" * 20)
    time.sleep(1)

    # --- TURN 3: Professional Llama ---
    # It reacts to the snarky comment
    print(f"🟢 Llama (Professional):")
    response_3 = ask_bot("llama3.2:1b", prompt_professional, response_2)
    print(response_3)
    print("-" * 20)
    time.sleep(1)

    # Update the topic for the next round so they keep talking about the last thing said
    current_message = response_3

--- STARTING THE CRAZY CHAT ---


=== ROUND 1 ===

🔴 Llama (Polite):
I'd be happy to help with writing Python code. As a gentle and careful assistant, I'll do my best to guide you through the process.

Before we start, could you please tell me what specific part of writing Python code you're struggling with
--------------------
🔵 Qwen (Snarky):
Ugh, writing Python? That's like asking for trouble wrapped in spaghetti code! Gentle guidance sounds good... Or did you mean someone who would rip your code apart and leave you bleeding on the coding floor?

Oh well, proceed. But just know
--------------------
🟢 Llama (Professional):
I stand in front of you, my eyes narrowing at the snarky tone and the sarcastic remark about my writing skills. I place a decisive hand on your arm, holding you in place.

Let me make one thing clear: I don't
--------------------

=== ROUND 2 ===

🔴 Llama (Polite):
*my voice remains steady with concern, but slightly trembles* I-I'd like to understand what's wrong, 